# Hoja de trabajo 2

- Mathew Alexander Cordero Aquino - 22982
- Gustavo Adolfo Cruz Bardales - 22779
- Edwin de Leon - 22809

https://github.com/donmatthiuz/VIC/tree/ht2

# Task 1 – Análisis

## 1. ¿Es correcto usar Homografías para medir la distancia mientras el robot gira?

No, no es un enfoque correcto, cuando el robot gira sobre su propio eje, el movimiento entre vistas corresponde a una rotación pura, lo que implica que los centros de cámara C1 y C2 coinciden (no hay traslación). En esta situación no se genera paralaje, que es el fenómeno necesario para estimar profundidad.

La estimación de distancia requiere que exista un baseline distinto de cero entre C1 y C2 para producir desplazamiento aparente de los puntos en la imagen. Durante una rotación pura:

- C1 ≈ C2  
- Baseline = 0  
- Paralaje ≈ 0  

Por ello, aunque una homografía puede modelar la relación geométrica entre las imágenes bajo rotación, no permite recuperar la distancia métrica (Z) de los objetos en una escena general del almacén.

**Conclusión:** El ingeniero junior está equivocado; sin traslación no hay paralaje y no es posible medir distancia usando homografías en este caso.


## 2. Si la disparidad (d) aumenta repentinamente, ¿qué se infiere sobre la distancia (Z)? ¿Qué riesgo industrial implica un error?

En visión estéreo se cumple:

\[
Z = \frac{f \cdot B}{d}
\]

Esto indica que la distancia Z es inversamente proporcional a la disparidad **d**.

Por lo tanto, si la disparidad de la caja aumenta repentinamente, se infiere que la distancia Z disminuye, es decir, la caja está más cerca del robot.

### Riesgo industrial de un error en la disparidad

Un cálculo incorrecto de la disparidad produce una estimación errónea de la distancia, lo que puede provocar:

- Colisiones del robot con cajas o estanterías  
- Fallos en el posicionamiento del sistema de agarre  
- Paradas de emergencia innecesarias  
- Daños al inventario o al robot  

**Conclusión:** Un aumento de disparidad indica que el objeto está más cercano, y errores en su cálculo representan un riesgo crítico para la navegación y operación segura del robot en el almacén.

## Task 2 – Ingeniería de Dimensiones

**1. Cálculo de las dimensiones ($W_{out}, H_{out}$)**



$$O = \left\lfloor \frac{W - F + 2P}{S} \right\rfloor + 1$$



Donde:


**W** (o **H**): Dimensión de la entrada.



**F**: Tamaño del filtro o Kernel.


**P**: Padding.


**S**: Stride.



Aplicando los valores dados para calcular el ancho ($W = 1280$, $F = 5$, $P = 2$, $S = 2$):

$$W_{out} = \left\lfloor \frac{1280 - 5 + 2(2)}{2} \right\rfloor + 1$$

$$W_{out} = \left\lfloor \frac{1280 - 5 + 4}{2} \right\rfloor + 1$$

$$W_{out} = \left\lfloor \frac{1279}{2} \right\rfloor + 1$$

$$W_{out} = \lfloor 639.5 \rfloor + 1$$

$$W_{out} = 639 + 1 = 640$$

Aplicando los valores para calcular el alto ($H = 720$, $F = 5$, $P = 2$, $S = 2$):

$$H_{out} = \left\lfloor \frac{720 - 5 + 2(2)}{2} \right\rfloor + 1$$

$$H_{out} = \left\lfloor \frac{720 - 5 + 4}{2} \right\rfloor + 1$$

$$H_{out} = \left\lfloor \frac{719}{2} \right\rfloor + 1$$

$$H_{out} = \lfloor 359.5 \rfloor + 1$$

$$H_{out} = 359 + 1 = 360$$

Respuesta: Las dimensiones del Mapa de Características resultante son **$640 \times 360$**.

**2. Cambio a Padding $P=0$**

Si cambiamos el Padding a $P=0$, las nuevas dimensiones de la salida se reducen de la siguiente manera:

$$W_{out} = \left\lfloor \frac{1280 - 5 + 0}{2} \right\rfloor + 1 = \lfloor 637.5 \rfloor + 1 = 637 + 1 = 638$$

$$H_{out} = \left\lfloor \frac{720 - 5 + 0}{2} \right\rfloor + 1 = \lfloor 357.5 \rfloor + 1 = 357 + 1 = 358$$

**¿Cómo afectaría esto a la información de los bordes?**
El propósito del padding es añadir píxeles en los bordes para permitir que el filtro procese la información de las esquinas. Si hacemos ese Valid Padding, el filtro no tendrá espacio para centrarse en los píxeles más extremos de la imagen de entrada. Como resultado, la información de los bordes se descartará gradualmente con cada capa.




## Task 3 – Criterio de Diseño

1. Usted está desarrollando un sistema de detección de grietas microscópicas en motores de avión.
¿Qué combinación de Stride y Pooling recomendaría para no perder detalles críticos en las
primeras capas de la red? Justifique técnicamente.


Recomendaría utilizar un stride de $S=1$ y evitar el uso de Pooling o usar un Average Pooling muy conservador en las primeras capas.

Justificación:
Un Stride de $S=1$ genera un movimiento denso del filtro. Esto asegura que el Kernel evalúe todas las vecindades de la imagen sin saltarse regiones pequeñas, lo cual es bueno para detectar anomalías diminutas. Si usara un $S=2$, el sistema saltaría cada dos píxeles y reduciría la resolución a la mitad, corriendo el riesgo de que la grieta desaparezca del mapa de características. Adicionalmente, las capas de Pooling están diseñadas para resumir y sub-muestrear la información. Aplicarlas demasiado pronto podría borrar la alta frecuencia antes de que las capas más profundas puedan abstraer su forma.

2. Un cliente le pide que el sistema funcione en un procesador muy limitado (como una cámara
inteligente con poca RAM). Explique cómo podrías utilizar el Stride y el Max Pooling
estratégicamente para reducir la carga computacional sin eliminar las características más fuertes
(activaciones) del Mapa de Características

La estrategia en este caso seria de utilizar una combinación de stride de $S=2$ y capas de max pooling.

Esto porque al aplicar un stride de $S=2$, obligamos al filtro a saltar posiciones, lo que reduce inmediatamente la resolución a la mitad en cada dimensión espacial, comprimiendo la información relevante rápidamente y disminuyendo drásticamente el tamaño del tensor. Esto alivia la memoria RAM y reduce la cantidad de operaciones en la siguiente capa.
Para asegurar que no se pierdan características clave, usamos el max pooling. Su función es tomar únicamente la activación más fuerte, el valor máximo, de una región. Esto significa que, aunque realizamos un sub-muestreo y descartamos parámetros, también garantizamos que los rasgos más distintivos que detectaron los filtros sobrevivan y pasen a la siguiente etapa de la red.

## Task 4 – Implementación Práctica

Con esta parte se busca que puedan comprender la mecánica interna de la operación convolucional sin
depender de librerías de alto nivel para la lógica central.Por ello realice lo siguiente:


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt



def manual_convolution(image, kernel, stride, padding):
    img = image.astype(np.float32)
    kernel = np.flipud(np.fliplr(kernel))


    k_h, k_w = kernel.shape
    h, w, c = img.shape

    img_with_padding = np.zeros([
        (h + padding * 2),
        (w + padding * 2),
        c
    ])

    r_pad = img_with_padding[:, :, 0]
    g_pad = img_with_padding[:, :, 1]
    b_pad = img_with_padding[:, :, 2]
    
    r_pad[padding : -padding, padding : -padding] = img[:, :, 0]

    g_pad[padding : -padding, padding : -padding] = img[:, :, 1]
    b_pad[padding : -padding, padding : -padding] = img[:, :, 2]
    
    fin_arr = np.dstack((r_pad, g_pad, b_pad))












